### What is Cache-Augmented Generation (CAG)?
CAG is a retrieval-free approach that bypasses the usual step of querying external knowledge sources at inference time. Instead, it preloads relevant documents into the LLM's extended context window, precomputes the model’s key‑value (KV) cache, and reuses this during inference—so the model can generate responses without additional retrieval steps 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

# LLm Model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5")
llm

c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001B55F9378C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B55FDCC440>, root_client=<openai.OpenAI object at 0x000001B55F9356A0>, root_async_client=<openai.AsyncOpenAI object at 0x000001B55FDCC1A0>, model_name='gpt-5', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [2]:
# Cache variable

Model_Cache = {}

In [3]:
import time

def cache_model(query):
    start_time = time.time()
    
    if Model_Cache.get(query):
        print("**CAche Hit**")
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"EXECUTION TIME: {elapsed_time:.2f} seconds")
        
        return Model_Cache.get(query)
    
    else:
        print("***CACHE MISS – EXECUTING MODEL***")
        start_time = time.time()
        response = llm.invoke(query)
        end_time = time.time()
        elapsed = end_time - start_time
        print(f"EXECUTION TIME: {elapsed:.2f} seconds")
        Model_Cache[query] = response
        
        return response

In [4]:
response = cache_model("hi")
response

***CACHE MISS – EXECUTING MODEL***
EXECUTION TIME: 3.07 seconds


AIMessage(content='Hi! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 7, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CIAv33Nn956frBtoGGe35IUMOGOF6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7a06c818-48db-476d-a75f-df913e5a78fc-0', usage_metadata={'input_tokens': 7, 'output_tokens': 18, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
Model_Cache

{'hi': AIMessage(content='Hi! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 7, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CIAv33Nn956frBtoGGe35IUMOGOF6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7a06c818-48db-476d-a75f-df913e5a78fc-0', usage_metadata={'input_tokens': 7, 'output_tokens': 18, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})}

In [6]:
response = cache_model("hi")
response

**CAche Hit**
EXECUTION TIME: 0.00 seconds


AIMessage(content='Hi! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 7, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CIAv33Nn956frBtoGGe35IUMOGOF6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7a06c818-48db-476d-a75f-df913e5a78fc-0', usage_metadata={'input_tokens': 7, 'output_tokens': 18, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
query = "can you give me 500 words on langgraph?"

response = cache_model(query)
print(response)

***CACHE MISS – EXECUTING MODEL***
EXECUTION TIME: 32.39 seconds
content='LangGraph is an open-source framework from the LangChain ecosystem for building stateful, multi-step, and multi-agent AI applications as graphs. Instead of treating an LLM call as a single, linear chain, LangGraph lets you model your system as nodes (functions) connected by edges (routing rules). Nodes read and update a shared state, and the graph’s scheduler executes them in a controlled loop, enabling complex behaviors like tool use, planning-execution cycles, error handling, and human-in-the-loop interruptions.\n\nAt the core are StateGraph and MessageGraph. A StateGraph is a typed state machine: you declare a state schema (often with Pydantic or TypedDict), define node functions that take the current state and return partial updates, and add edges that determine which node runs next. MessageGraph is a convenience for chat-centric apps, treating state as a message list plus metadata. LangGraph supports conditi

In [8]:
Model_Cache

{'hi': AIMessage(content='Hi! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 7, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CIAv33Nn956frBtoGGe35IUMOGOF6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7a06c818-48db-476d-a75f-df913e5a78fc-0', usage_metadata={'input_tokens': 7, 'output_tokens': 18, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 'can you give me 500 words on langgraph?': AIMessage(content='LangGraph is an open-source framework from the LangChain ecosystem for building stateful, multi-step, and multi-agent A

In [9]:
query = "can you give me 500 words on langgraph?"

response = cache_model(query)
print(response)

**CAche Hit**
EXECUTION TIME: 0.00 seconds
content='LangGraph is an open-source framework from the LangChain ecosystem for building stateful, multi-step, and multi-agent AI applications as graphs. Instead of treating an LLM call as a single, linear chain, LangGraph lets you model your system as nodes (functions) connected by edges (routing rules). Nodes read and update a shared state, and the graph’s scheduler executes them in a controlled loop, enabling complex behaviors like tool use, planning-execution cycles, error handling, and human-in-the-loop interruptions.\n\nAt the core are StateGraph and MessageGraph. A StateGraph is a typed state machine: you declare a state schema (often with Pydantic or TypedDict), define node functions that take the current state and return partial updates, and add edges that determine which node runs next. MessageGraph is a convenience for chat-centric apps, treating state as a message list plus metadata. LangGraph supports conditional routing, parallel

In [10]:
query = "give me 500 words on langgraph?"

response = cache_model(query)
print(response)

***CACHE MISS – EXECUTING MODEL***
EXECUTION TIME: 26.37 seconds
content='LangGraph is an open-source framework for building stateful, reliable AI workflows as graphs. It extends the LangChain ecosystem by modeling an application as a set of nodes (functions, models, tools) connected by edges that encode control flow and state transitions. Instead of writing linear chains or ad‑hoc agent loops, you describe a finite state machine: the graph reads and writes a shared state object and routes execution based on that state. The result is determinism where you want it, flexibility where you need it, and observability throughout.\n\nCore ideas:\n- State: a typed dictionary that persists across nodes. You declare reducers (how updates merge) and can checkpoint or resume.\n- Nodes: callables that transform state, including LLM calls, tool invocations, or custom code. Nodes are pure with respect to inputs/outputs, which makes testing and caching straightforward.\n- Edges: conditional routing th

### Advanced CAG

In [11]:
from __future__ import annotations
from typing import TypedDict, List, Optional
import time

# ---- LangGraph / LangChain ----
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain.embeddings import HuggingFaceEmbeddings

# ---- FAISS vector stores ----
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [12]:
# ================= CONFIG =================
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dim
VECTOR_DIM = 384

LLM_MODEL = "gpt-4o-mini"
LLM_TEMPERATURE = 0

RETRIEVE_TOP_K = 4
CACHE_TOP_K = 3

CACHE_DISTANCE_THRESHOLD = 0.45

# Optional TTL for cache entries (seconds). 0 = disabled.
CACHE_TTL_SEC = 0

In [13]:
# ================= STATE ==================
class RAGState(TypedDict):
    question: str
    normalized_question: str
    context_docs: List[Document]
    answer: Optional[str]
    citations: List[str]
    cache_hit: bool

In [14]:
# ============== GLOBALS ===================
from langchain_huggingface import HuggingFaceEmbeddings
EMBED = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


In [15]:
# ----- QA CACHE (EMPTY, SAFE INIT) -----
qa_index = faiss.IndexFlatL2(VECTOR_DIM)  # distance; lower is better
QA_CACHE = FAISS(
    embedding_function=EMBED,
    index=qa_index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

In [16]:
QA_CACHE

In [17]:
# ----- RAG STORE (demo only) -----
RAG_STORE = FAISS.from_texts(
    texts=[
        "LangGraph lets you compose stateful LLM workflows as graphs.",
        "In LangGraph, nodes can be cached; node caching memoizes outputs keyed by inputs for a TTL.",
        "Retrieval-Augmented Generation (RAG) retrieves external context and injects it into prompts.",
        "Semantic caching reuses prior answers when new questions are semantically similar."
    ],
    embedding=EMBED,
)

In [18]:
LLM = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)
LLM

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001B5A6196210>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B5A61965D0>, root_client=<openai.OpenAI object at 0x000001B5A6195F90>, root_async_client=<openai.AsyncOpenAI object at 0x000001B5A6196350>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [19]:
# ================ NODES ===================
def normalize_query(state: RAGState) -> RAGState:
    q = (state["question"] or "").strip()
    state["normalized_question"] = q.lower()
    return state

def semantic_cache_lookup(state: RAGState) -> RAGState:
    q = state["normalized_question"]
    state["cache_hit"] = False  # default

    if not q:
        return state

    # ✅ Guard: FAISS crashes if ntotal == 0 and you ask for k>0
    if getattr(QA_CACHE, "index", None) is None or QA_CACHE.index.ntotal == 0:
        return state

    # For FAISS L2 wrapper, this returns (Document, distance) with lower=better
    hits = QA_CACHE.similarity_search_with_score(q, k=CACHE_TOP_K)
    if not hits:
        return state

    best_doc, dist = hits[0]

    # Optional TTL
    if CACHE_TTL_SEC > 0:
        ts = best_doc.metadata.get("ts")
        if ts is None or (time.time() - float(ts)) > CACHE_TTL_SEC:
            return state

    # L2 distance gate (lower = more similar)
    if dist <= CACHE_DISTANCE_THRESHOLD:
        cached_answer = best_doc.metadata.get("answer")
        if cached_answer:
            state["answer"] = cached_answer
            state["citations"] = ["(cache)"]
            state["cache_hit"] = True

    return state

def respond_from_cache(state: RAGState) -> RAGState:
    return state

def retrieve(state: RAGState) -> RAGState:
    q = state["normalized_question"]
    docs = RAG_STORE.similarity_search(q, k=RETRIEVE_TOP_K)
    state["context_docs"] = docs
    return state

def generate(state: RAGState) -> RAGState:
    q = state["question"]
    docs = state.get("context_docs", [])
    ctx = "\n\n".join([f"[doc-{i}] {d.page_content}" for i, d in enumerate(docs, start=1)])

    system = (
        "You are a precise RAG assistant. Use the context when helpful. "
        "Cite with [doc-i] markers if you use a fact from the context."
    )
    user = f"Question: {q}\n\nContext:\n{ctx}\n\nWrite a concise answer with citations."

    resp = LLM.invoke([{"role": "system", "content": system},
                       {"role": "user", "content": user}])
    state["answer"] = resp.content
    state["citations"] = [f"[doc-{i}]" for i in range(1, len(docs) + 1)]
    return state

def cache_write(state: RAGState) -> RAGState:
    q = state["normalized_question"]
    a = state.get("answer")
    if not q or not a:
        return state

    QA_CACHE.add_texts(
        texts=[q],
        metadatas=[{
            "answer": a,
            "ts": time.time(),
        }]
    )
    return state

In [20]:
# ============== GRAPH WIRING ==============
graph = StateGraph(RAGState)

graph.add_node("normalize_query", normalize_query)
graph.add_node("semantic_cache_lookup", semantic_cache_lookup)
graph.add_node("respond_from_cache", respond_from_cache)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("cache_write", cache_write)

graph.set_entry_point("normalize_query")
graph.add_edge("normalize_query", "semantic_cache_lookup")

def _branch(state: RAGState) -> str:
    return "respond_from_cache" if state.get("cache_hit") else "retrieve"

graph.add_conditional_edges(
    "semantic_cache_lookup",
    _branch,
    {
        "respond_from_cache": "respond_from_cache",
        "retrieve": "retrieve"
    }
)

graph.add_edge("respond_from_cache", END)
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "cache_write")
graph.add_edge("cache_write", END)

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

In [21]:
# ================= DEMO ===================
if __name__ == "__main__":
    thread_cfg = {"configurable": {"thread_id": "demo-user-1"}}

    q1 = "What is LangGraph ?"
    out1 = app.invoke({"question": q1, "context_docs": [], "citations": []}, thread_cfg)
    print("Answer:", out1["answer"])
    print("Citations:", out1.get("citations"))
    print("Cache hit?:", out1.get("cache_hit"))

Answer: LangGraph is a framework that allows users to compose stateful workflows for large language models (LLMs) in the form of graphs, enabling the management of complex interactions and data flows [doc-2]. It also features node caching, which memoizes outputs based on inputs for a specified time-to-live (TTL) [doc-1].
Citations: ['[doc-1]', '[doc-2]', '[doc-3]', '[doc-4]']
Cache hit?: False


In [22]:
q1 = "Explain about LangGraph ?"
out1 = app.invoke({"question": q1, "context_docs": [], "citations": []}, thread_cfg)
print("Answer:", out1["answer"])
print("Citations:", out1.get("citations"))
print("Cache hit?:", out1.get("cache_hit"))

Answer: LangGraph is a framework that allows users to compose stateful workflows for large language models (LLMs) in the form of graphs, enabling the management of complex interactions and data flows [doc-2]. It also features node caching, which memoizes outputs based on inputs for a specified time-to-live (TTL) [doc-1].
Citations: ['(cache)']
Cache hit?: True


In [23]:
q1 = "Explain about LangGraph agents ?"
out1 = app.invoke({"question": q1, "context_docs": [], "citations": []}, thread_cfg)
print("Answer:", out1["answer"])
print("Citations:", out1.get("citations"))
print("Cache hit?:", out1.get("cache_hit"))

Answer: LangGraph agents are part of a framework that allows users to compose stateful workflows using large language models (LLMs) structured as graphs [doc-2]. These agents can utilize features like node caching, which memoizes outputs based on inputs for a specified time-to-live (TTL) [doc-1]. Additionally, they can leverage semantic caching to reuse previous answers when new queries are semantically similar, enhancing efficiency [doc-4]. This approach is particularly useful in Retrieval-Augmented Generation (RAG), where external context is retrieved and integrated into prompts to improve response quality [doc-3].
Citations: ['[doc-1]', '[doc-2]', '[doc-3]', '[doc-4]']
Cache hit?: False


In [24]:
q1 = "Explain about agents in Langgraph ?"
out1 = app.invoke({"question": q1, "context_docs": [], "citations": []}, thread_cfg)
print("Answer:", out1["answer"])
print("Citations:", out1.get("citations"))
print("Cache hit?:", out1.get("cache_hit"))

Answer: LangGraph agents are part of a framework that allows users to compose stateful workflows using large language models (LLMs) structured as graphs [doc-2]. These agents can utilize features like node caching, which memoizes outputs based on inputs for a specified time-to-live (TTL) [doc-1]. Additionally, they can leverage semantic caching to reuse previous answers when new queries are semantically similar, enhancing efficiency [doc-4]. This approach is particularly useful in Retrieval-Augmented Generation (RAG), where external context is retrieved and integrated into prompts to improve response quality [doc-3].
Citations: ['(cache)']
Cache hit?: True
